# Bài 39: Trực quan hóa và Thấu hiểu CNN

Một CNN đã huấn luyện là một hộp đen vì hàng triệu trọng số của nó không có ý nghĩa riêng lẻ rõ ràng. Nhưng câu hỏi một dự đoán xuất phát từ *đâu trong ảnh đầu vào* thì trả lời được, và trả lời được nó thường là điều phân biệt "mô hình đưa ra đáp án đúng" với "mô hình đưa ra đáp án đúng vì đúng lý do". Bài này xây dựng từ đầu hai công cụ trực quan hóa — **bản đồ nổi bật (saliency maps)** (<a href="../references.html#simonyan-2013-saliency">Simonyan et al., 2013</a>) và **Grad-CAM** (<a href="../references.html#selvaraju-2017">Selvaraju et al., 2017</a><span class="landmark-paper">&#9733;</span>) — trên ảnh chụp thật từ CIFAR-10 (<a href="../references.html#krizhevsky-2009-cifar">Krizhevsky, 2009</a>), rồi dùng chúng để bắt quả tang một mô hình đang gian lận. Cuối cùng, **t-SNE** (<a href="../references.html#vandermaaten-2008-tsne">Van der Maaten & Hinton, 2008</a><span class="landmark-paper">&#9733;</span>) được dùng để trực quan hóa các vector đặc trưng học được.

In [ ]:
import pickle
import tarfile
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Thiết lập: một CNN mèo-và-ô-tô, với các bản đồ đặc trưng được lộ ra

Một nhiệm vụ nhị phân trên CIFAR-10: mèo và ô tô, mỗi lớp 300 ảnh huấn luyện. Kiến trúc theo mẫu CNN ở Bài 34, ngoại trừ `forward` giờ còn trả về bản đồ đặc trưng của lớp tích chập cuối cùng (trước phép gộp toàn cục), để cả hai phương pháp trực quan hóa đều truy cập được nó.

In [ ]:
CIFAR_URL = 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'
CACHE_ROOT = Path.home() / '.cache' / 'cvintro'
CACHE_DIR = CACHE_ROOT / 'cifar-10-batches-py'

def ensure_cifar10():
    if CACHE_DIR.exists():
        return
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    archive_path = CACHE_ROOT / 'cifar-10-python.tar.gz'
    if not archive_path.exists():
        print('Downloading CIFAR-10 (~163 MB, one-time, cached under ~/.cache/cvintro)...')
        urllib.request.urlretrieve(CIFAR_URL, archive_path)
    print('Extracting...')
    with tarfile.open(archive_path) as tar:
        tar.extractall(CACHE_ROOT)

def load_cifar_batch(path):
    with open(path, 'rb') as f:
        d = pickle.load(f, encoding='bytes')
    imgs = d[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.float32) / 255.0
    labels = np.array(d[b'labels'], dtype=np.int64)
    return imgs, labels

ensure_cifar10()

CIFAR_LABELS = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
CID = {name: CIFAR_LABELS.index(name) for name in CIFAR_LABELS}

train_imgs, train_labels = [], []
for i in range(1, 6):
    imgs, labels = load_cifar_batch(CACHE_DIR / f'data_batch_{i}')
    train_imgs.append(imgs); train_labels.append(labels)
train_imgs, train_labels = np.concatenate(train_imgs), np.concatenate(train_labels)
test_imgs, test_labels = load_cifar_batch(CACHE_DIR / 'test_batch')

def take(imgs, labels, name, n, rng_local):
    idx = np.where(labels == CID[name])[0]
    idx = rng_local.permutation(idx)[:n]
    return imgs[idx].copy()

rng = np.random.default_rng(4)
X_cat_train = take(train_imgs, train_labels, 'cat', 300, rng)
X_auto_train = take(train_imgs, train_labels, 'automobile', 300, rng)
X_cat_test = take(test_imgs, test_labels, 'cat', 100, rng)
X_auto_test = take(test_imgs, test_labels, 'automobile', 100, rng)

X_train = np.concatenate([X_cat_train, X_auto_train])
y_train = np.array([0.0] * 300 + [1.0] * 300, dtype=np.float32)
X_test = np.concatenate([X_cat_test, X_auto_test])
y_test = np.array([0.0] * 100 + [1.0] * 100, dtype=np.float32)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, 5, padding=2)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(8, 16, 5, padding=2)
        self.gpool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        f1 = F.relu(self.conv1(x))  # 32x32, full input resolution
        f2 = F.relu(self.conv2(self.pool(f1)))  # last conv feature map, downsampled to 16x16
        feat = self.gpool(f2).flatten(1)
        return self.fc(feat).squeeze(-1), f2

torch.manual_seed(0)
model_original = CNN()
opt = torch.optim.Adam(model_original.parameters(), lr=0.001)
Xt = torch.tensor(X_train).permute(0, 3, 1, 2); yt = torch.tensor(y_train)
for _ in range(300):
    opt.zero_grad()
    out, _ = model_original(Xt)
    loss = F.binary_cross_entropy_with_logits(out, yt)
    loss.backward()
    opt.step()

with torch.no_grad():
    out, _ = model_original(torch.tensor(X_test).permute(0, 3, 1, 2))
    acc = ((out > 0).float() == torch.tensor(y_test)).float().mean().item()
print(f'test accuracy: {acc:.1%}')

## Bản đồ nổi bật

Ý tưởng của bản đồ nổi bật (<a href="../references.html#simonyan-2013-saliency">Simonyan et al., 2013</a>): lấy gradient của *điểm số* lớp được dự đoán theo mọi điểm ảnh đầu vào. Một điểm ảnh có gradient độ lớn cao là điểm mà một thay đổi nhỏ sẽ làm thay đổi dự đoán nhiều nhất — tức là một điểm ảnh mà mạng đang "nhìn vào".

In [ ]:
def saliency_map(model, img_hw3):
    x = torch.tensor(img_hw3).permute(2, 0, 1).unsqueeze(0)
    x.requires_grad_(True)
    score, feat = model(x)
    score.backward()
    return x.grad[0].abs().amax(dim=0).numpy(), feat  # max abs gradient across the 3 color channels

idx = 3
saliency, _ = saliency_map(model_original, X_test[idx])

fig, axes = plt.subplots(1, 2, figsize=(7, 3.2))
axes[0].imshow(X_test[idx])
axes[0].set_title('input image'); axes[0].axis('off')
axes[1].imshow(saliency, cmap='hot')
axes[1].set_title('saliency map')
axes[1].axis('off')
plt.show()

## Grad-CAM

Bản đồ nổi bật thô ở cấp điểm ảnh và có xu hướng nhiễu. **Grad-CAM** (<a href="../references.html#selvaraju-2017">Selvaraju et al., 2017</a><span class="landmark-paper">&#9733;</span>) thay vào đó hoạt động trên các bản đồ đặc trưng của một lớp tích chập được chọn, thường là lớp tích chập cuối cùng. Dù các bản đồ đặc trưng này có độ phân giải thấp hơn một bản đồ nổi bật cấp điểm ảnh (bị giảm mẫu bởi các lớp gộp phía trước), chúng có xu hướng mang nhiều thông tin ngữ nghĩa phân biệt lớp hơn so với các gradient điểm ảnh thô riêng lẻ. Phương pháp:

1. Tính gradient của điểm số lớp đích theo mỗi kênh bản đồ đặc trưng.
2. Lấy trung bình toàn cục gradient của mỗi kênh trên các chiều không gian để có một trọng số quan trọng cho mỗi kênh.
3. Lấy tổng có trọng số của các kênh bản đồ đặc trưng và áp dụng ReLU, giữ lại các đóng góp dương cho lớp đích.

Bản đồ nhiệt (heatmap) thô thu được có độ phân giải không gian của lớp tích chập đã chọn; nó có thể được tăng mẫu lên độ phân giải đầu vào để trực quan hóa.

In [ ]:
def grad_cam(model, img_hw3, out_size=32):
    x = torch.tensor(img_hw3).permute(2, 0, 1).unsqueeze(0)
    x.requires_grad_(True)
    score, feat = model(x)
    feat.retain_grad()
    score.backward()
    weights = feat.grad[0].mean(dim=(1, 2))  # (channels,) importance per channel
    cam = F.relu((weights[:, None, None] * feat[0]).sum(dim=0))
    cam_up = F.interpolate(cam[None, None], size=(out_size, out_size), mode='bilinear', align_corners=False)
    return cam_up[0, 0].detach().numpy()

cam = grad_cam(model_original, X_test[idx])

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
axes[0].imshow(X_test[idx])
axes[0].set_title('input image'); axes[0].axis('off')
axes[1].imshow(saliency, cmap='hot')
axes[1].set_title('saliency map'); axes[1].axis('off')
axes[2].imshow(cam, cmap='hot')
axes[2].set_title('Grad-CAM'); axes[2].axis('off')
plt.show()

Ảnh động bên dưới giúp dễ thấy sự tương phản hơn: bản đồ nổi bật cấp điểm ảnh cạnh bản đồ nhiệt Grad-CAM thô hơn, mang tính ngữ nghĩa hơn, cả hai trên cùng một ảnh.

<img src="../animations/saliency_gradcam_demo.gif" alt="Ảnh động hiển thị dần bản đồ nổi bật và bản đồ nhiệt Grad-CAM hiện lên cạnh nhau trên cùng một ảnh đầu vào, cho thấy khả năng định vị cấp điểm ảnh so với định vị ngữ nghĩa thô hơn của hai phương pháp" style="display: block; margin: 0 auto; max-width: 820px;">

## Các bản đồ này có thực sự theo dõi thứ mà mô hình sử dụng không?

Để kiểm tra xem các công cụ này có hữu ích không, hãy cố tình cho mô hình một lối tắt, rồi kiểm tra xem chúng có bắt đúng được nó không. Thêm một dấu hiệu trắng nhỏ 5x5 không thể nhầm lẫn vào góc trên-trái của chỉ mọi ảnh huấn luyện **ô tô** (không bao giờ trên ảnh mèo) — đại diện cho một yếu tố gây nhiễu (confound) thực tế, như một hình mờ (watermark), một nhiễu đặc thù của phòng thí nghiệm, hoặc một đặc điểm kỳ quặc của thiết bị chụp tình cờ tương quan với một lớp. Huấn luyện một mô hình thứ hai, `model_shortcut`, trên tập dữ liệu bị làm hỏng này, và so sánh nó với `model_original` ban đầu ở trên (chưa bao giờ tiếp xúc với bất kỳ dấu hiệu nào).

In [ ]:
def add_marker(imgs):
    out = imgs.copy()
    out[:, 0:5, 0:5, :] = 1.0  # a stark 5x5 white square, top-left corner
    return out

X_train_shortcut = np.concatenate([X_cat_train, add_marker(X_auto_train)])
X_auto_test_marked = add_marker(X_auto_test)
X_test_marked = np.concatenate([X_cat_test, X_auto_test_marked])  # marker present at test time too

torch.manual_seed(0)
model_shortcut = CNN()
opt2 = torch.optim.Adam(model_shortcut.parameters(), lr=0.001)
Xt_shortcut = torch.tensor(X_train_shortcut).permute(0, 3, 1, 2)
for _ in range(300):
    opt2.zero_grad()
    out, _ = model_shortcut(Xt_shortcut)
    loss = F.binary_cross_entropy_with_logits(out, torch.tensor(y_train))
    loss.backward()
    opt2.step()

def acc_of(m, X, y):
    with torch.no_grad():
        out, _ = m(torch.tensor(X).permute(0, 3, 1, 2))
        return ((out > 0).float() == torch.tensor(y)).float().mean().item()

print(f'{"":>16} {"clean test":>12} {"marked test":>13}')
print(f'{"model_original":>16} {acc_of(model_original, X_test, y_test):>11.1%} {acc_of(model_original, X_test_marked, y_test):>13.1%}')
print(f'{"model_shortcut":>16} {acc_of(model_shortcut, X_test, y_test):>11.1%} {acc_of(model_shortcut, X_test_marked, y_test):>13.1%}')

Độ chính xác của `model_original` hầu như không thay đổi dù có dấu hiệu hay không — vì nó chưa bao giờ học cách dùng nó, nó chẳng mất gì khi dấu hiệu vắng mặt.

`model_shortcut` trông *tốt hơn* `model_original` khi có dấu hiệu, nhưng nó sụp đổ khi dấu hiệu bị loại bỏ. Nó đã học lối tắt "ô vuông trắng ở góc = ô tô".

Giờ hãy dùng bản đồ nổi bật và Grad-CAM để kiểm tra xem các công cụ trực quan hóa có thực sự bắt được điều này không. Vì vị trí của dấu hiệu được biết chính xác (hàng 0-4, cột 0-4), phép kiểm tra "khoảng cách tới tâm thực" từ một tập dữ liệu tổng hợp được kiểm soát trở thành: với mỗi phương pháp, đỉnh bản đồ nhiệt của mỗi mô hình rơi vào bên trong đúng vùng 5x5 đó ở bao nhiêu phần trăm ảnh kiểm tra?

In [ ]:
def peak_in_marker_frac(method, m, imgs):
    count = 0
    for img in imgs:
        heatmap = saliency_map(m, img)[0] if method == 'saliency' else grad_cam(m, img)
        peak = np.unravel_index(heatmap.argmax(), heatmap.shape)  # (row, col)
        if peak[0] < 5 and peak[1] < 5:
            count += 1
    return count / len(imgs)

rows = [f'{"model":>16} {"test images":>20} {"saliency":>10} {"Grad-CAM":>10}']
for name, m, imgs in [('model_original', model_original, X_auto_test_marked),
                       ('model_shortcut', model_shortcut, X_auto_test_marked),
                       ('model_shortcut', model_shortcut, X_auto_test)]:
    label = 'marked' if imgs is not X_auto_test else 'clean'
    sal_frac = peak_in_marker_frac('saliency', m, imgs)
    cam_frac = peak_in_marker_frac('grad_cam', m, imgs)
    rows.append(f'{name:>16} {label:>20} {sal_frac:>9.1%} {cam_frac:>10.1%}')
print('\n'.join(rows))

`model_original` (chưa bao giờ được huấn luyện với dấu hiệu) không bao giờ có đỉnh của phương pháp nào rơi vào đúng góc đó. `model_shortcut` thì khác hẳn: trên các ảnh có dấu hiệu, cả hai công cụ gần như hoàn hảo (bản đồ nổi bật 96%, Grad-CAM 100%), nhưng trên các ảnh *sạch* — nơi dấu hiệu chưa bao giờ được thêm vào, nhưng `model_shortcut` vẫn dựa vào nó bên trong — đỉnh của Grad-CAM vẫn nhất quán rơi vào đó 92% số lần, trong khi bản đồ nổi bật giảm mạnh xuống 56%. Đó là vì Grad-CAM gộp gradient trên cả một kênh bản đồ đặc trưng trước khi định vị, làm trơn nhiễu cấp điểm ảnh và khiến nó trở thành bộ phát hiện đáng tin cậy hơn khi tác nhân kích hoạt theo nghĩa đen của lối tắt không hiển thị trong một ảnh nhất định. Ngược lại, bộ máy bên trong của `model_shortcut` bị neo vào vị trí đó bất kể thực sự có gì ở đó, chính là kiểu thất bại được mô tả ở đầu bài.

Ảnh động bên dưới xây dựng bảng ở trên, giúp dễ so sánh cả hai phương pháp trên cùng các ảnh, đánh dấu một hồng tâm tại đúng điểm ảnh đỉnh của mỗi bản đồ nhiệt và việc nó có rơi vào bên trong vùng dấu hiệu trên-trái hay không, với số đếm tích lũy của mỗi khung hội tụ về tỉ lệ phần trăm trên toàn bộ tập kiểm tra.

<img src="../animations/shortcut_detection_demo.gif" alt="Ảnh động lưới 2x3, các hàng được gắn nhãn bản đồ nổi bật và Grad-CAM với các thẻ tô bóng, các cột là model_original trên ảnh có dấu hiệu, model_shortcut trên ảnh có dấu hiệu, và model_shortcut trên ảnh sạch; mỗi khung cho thấy bản đồ nhiệt của phương pháp đó trên một ảnh ví dụ với vùng dấu hiệu 5x5 đã biết được viền lại, một hồng tâm trắng-đen đánh dấu đúng điểm ảnh đỉnh, một viền màu đánh dấu đỉnh có rơi vào bên trong dấu hiệu hay không, và một dòng chữ hiển thị tỉ lệ phần trăm tích lũy và cuối cùng, hội tụ về lần lượt 0/0, 96/100 và 56/92 phần trăm cho bản đồ nổi bật/Grad-CAM" style="display: block; margin: 0 auto; max-width: 900px;">

Bài học thực tế: chỉ riêng độ chính xác (Bài 36) không thể phân biệt mô hình đã học tín hiệu thật hay một lối tắt tương quan với nó trong dữ liệu huấn luyện. Các công cụ trực quan hóa như bản đồ nổi bật và Grad-CAM thì có thể — nhưng chỉ khi bạn biết phải kiểm tra, nếu bạn chọn một công cụ đủ vững để tin tưởng, và nếu bạn diễn giải kết quả cẩn thận để nhận ra còn thứ gì khác trong ảnh có thể đang thu hút sự chú ý của gradient.

## Trực quan hóa không gian đặc trưng: t-SNE

Bản đồ nổi bật và Grad-CAM đều trả lời một câu hỏi *không gian* về mỗi lần một ảnh: mạng đang nhìn vào đâu? **t-SNE** (<a href="../references.html#vandermaaten-2008-tsne">van der Maaten & Hinton, 2008</a><span class="landmark-paper">&#9733;</span>) đặt một câu hỏi rộng hơn trên nhiều ảnh: biểu diễn bên trong của mạng có tách được các lớp mà nó được huấn luyện để nhận dạng không? Nó lấy vector đặc trưng 16 chiều của mỗi ảnh—biểu diễn được đưa trực tiếp vào bộ phân loại—và chiếu các vector này xuống 2D trong khi bảo toàn các vùng lân cận cục bộ. Khác với PCA (Bài 6), vốn dùng một phép chiếu tuyến tính, t-SNE nhấn mạnh cấu trúc cục bộ, khiến nó hữu ích để kiểm tra xem các ảnh cùng lớp có tụ lại với nhau trong không gian đặc trưng không mà không dùng nhãn của chúng trong phép chiếu.

In [ ]:
def tsne(X, n_iter=500, perplexity=15.0, lr=100.0, seed=0):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    sq_dists = ((X[:, None, :] - X[None, :, :]) ** 2).sum(-1)

    # high-dimensional affinities P: for each point, binary-search a Gaussian bandwidth so its
    # neighbor distribution has the target perplexity (an implicit "how many neighbors matter" knob)
    target_entropy = np.log(perplexity)
    P = np.zeros((n, n))
    for i in range(n):
        others = np.arange(n) != i
        d_i = sq_dists[i, others]
        lo, hi = 1e-4, 1e4
        for _ in range(50):
            beta = (lo + hi) / 2
            p = np.exp(-d_i * beta)
            p /= p.sum() + 1e-12
            entropy = -np.sum(p * np.log(p + 1e-12))
            if entropy > target_entropy:
                lo = beta  # entropy too high (too spread out) -> need a larger beta to sharpen it
            else:
                hi = beta
        P[i, others] = p
    P = (P + P.T) / (2 * n)          # symmetrize into one joint distribution over pairs
    P = np.maximum(P, 1e-12) * 4.0   # early exaggeration: temporarily inflate P so true neighbors clump together faster

    # low-dimensional embedding Y, fit by gradient descent on KL(P || Q)
    Y = rng.normal(0, 1e-2, (n, 2))
    velocity = np.zeros_like(Y)
    for it in range(n_iter):
        if it == 100:
            P /= 4.0  # turn off early exaggeration once clusters have separated
        d = ((Y[:, None, :] - Y[None, :, :]) ** 2).sum(-1)
        num = 1.0 / (1.0 + d)  # Student-t kernel -- heavy tails let moderately-distant points repel more strongly than a Gaussian would, avoiding the "crowding problem"
        np.fill_diagonal(num, 0)
        Q = np.maximum(num / num.sum(), 1e-12)
        coeff = (P - Q) * num
        grad = 4 * (coeff[:, :, None] * (Y[:, None, :] - Y[None, :, :])).sum(1)
        momentum = 0.5 if it < 100 else 0.8
        velocity = momentum * velocity - lr * grad
        Y = Y + velocity
    return Y

In [ ]:
def pooled_features(m, X):
    with torch.no_grad():
        _, f2 = m(torch.tensor(X).permute(0, 3, 1, 2))
        return F.adaptive_max_pool2d(f2, 1).flatten(1).numpy()  # same pooling model.gpool does internally

feat_trained = pooled_features(model_original, X_test)

torch.manual_seed(1)  # a second network, same architecture, never trained -- the baseline
untrained_model = CNN()
feat_untrained = pooled_features(untrained_model, X_test)

Y_trained = tsne(feat_trained, seed=0)
Y_untrained = tsne(feat_untrained, seed=0)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, Y, title in zip(axes, [Y_untrained, Y_trained], ['untrained CNN features', 'trained CNN features']):
    ax.scatter(*Y[:100].T, c='tab:blue', s=14, label='cat')
    ax.scatter(*Y[100:].T, c='tab:orange', s=14, label='automobile')
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7)
fig.suptitle("t-SNE of each test image's 16-d pooled feature vector, colored by true class", y=1.02)
plt.show()

Hình tĩnh ở trên chỉ cho thấy t-SNE kết thúc ở đâu. Ảnh động bên dưới cho thấy nó đến đó bằng cách nào: các điểm bắt đầu ở vị trí khởi tạo ngẫu nhiên và được hiển thị hội tụ, qua từng vòng lặp, thành hai cụm lớp.

<img src="../animations/tsne_convergence_demo.gif" alt="Ảnh động tối ưu hóa t-SNE: các điểm biểu diễn đặc trưng CNN đã huấn luyện của các ảnh kiểm tra mèo và ô tô bắt đầu tụ ngẫu nhiên gần gốc tọa độ rồi dần trải ra và tách thành hai vùng tụ lỏng lẻo khi hạ gradient tiến hành qua 500 vòng lặp" style="display: block; margin: 0 auto; max-width: 620px;">

Các đặc trưng của mạng chưa huấn luyện rải hai lớp lẫn lộn vào nhau không có cấu trúc nào thấy được — không ngạc nhiên, vì các trọng số tích chập của nó vẫn ngẫu nhiên và chưa bao giờ thấy một mẫu có nhãn nào. Các đặc trưng của mạng đã huấn luyện tạo thành hai cụm tách biệt thấy rõ, mèo và ô tô, dù bản thân t-SNE chưa bao giờ được cho biết điểm nào thuộc lớp nào; nó chỉ thấy các vector đặc trưng 16 chiều và khoảng cách từng cặp giữa chúng. (Về mặt định lượng, một phép kiểm tra láng giềng gần nhất đơn giản trong không gian nhúng 2D &mdash; láng giềng gần nhất của mỗi điểm có cùng nhãn thực với nó không? &mdash; cho kết quả 78% với đặc trưng đã huấn luyện so với khoảng 62% với đặc trưng chưa huấn luyện.) Đây là bằng chứng cho thấy huấn luyện không chỉ điều chỉnh lớp tuyến tính cuối cùng — nó định hình lại toàn bộ không gian đặc trưng sao cho mèo và ô tô thực sự sống ở các vùng lân cận khác nhau trong đó.

### Bài tập

1. Grad-CAM ở đây dùng lớp tích chập *cuối cùng*. Hãy sửa `grad_cam` để thay vào đó dùng bản đồ đặc trưng trung gian sau `conv1` (trước phép gộp và `conv2`). Bản đồ nhiệt thu được sắc nét hơn (gần hoàn hảo đến từng điểm ảnh hơn, giống bản đồ nổi bật) hay thô hơn, và tại sao một lớp sớm hơn lại hành xử như vậy?
2. Thu nhỏ dấu hiệu từ 5x5 xuống 2x2, hoặc làm mờ nó từ trắng thuần (`1.0`) xuống xám nhạt (`0.6`). `model_shortcut` có còn học cách dựa vào nó mạnh như vậy không (kiểm tra khoảng cách độ chính xác giữa ảnh sạch và ảnh có dấu hiệu), và Grad-CAM có còn bắt được nó một cách đáng tin cậy không?
3. Gradient của bản đồ nổi bật trong bài này được lấy theo logit thô (`score`), không phải xác suất sigmoid. Hãy thử tính nó theo `torch.sigmoid(score)` thay vào đó — bản đồ thu được có trông khác biệt đáng kể không, và bạn có thể giải thích tại sao bằng quy tắc chuỗi không?
4. Tô màu đồ thị t-SNE của đặc trưng đã huấn luyện theo nhãn *dự đoán* thay vì nhãn thực (`(model_original(...) > 0)` từ phần trước). Nhúm điểm rơi vào "phía sai" của ranh giới cụm có tương ứng với các ảnh mà mô hình thực sự phân loại sai không? Sau đó lặp lại với đặc trưng của `model_shortcut` trên `X_auto_test_marked` — việc loại bỏ dấu hiệu (`X_auto_test`, không có dấu hiệu) có làm sụp đổ sự tách lớp gọn gàng như các con số độ chính xác trước đó đã dự đoán không?